In [ ]:
import mlflow
import os 
from getpass import getpass
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Input
from tensorflow.keras.optimizers import RMSprop, SGD, Adam
from tensorflow.keras import regularizers
from keras.callbacks import ModelCheckpoint, EarlyStopping
import mlflow.keras
from keras.regularizers import l1_l2
from keras.regularizers import l1
from keras.regularizers import l2


dataset=mnist.load_data()
(x_train, y_train), (x_test, y_test) = dataset

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')

x_train /= 255  
x_test /= 255

num_classes=10
y_trainc = keras.utils.to_categorical(y_train, num_classes)
y_testc = keras.utils.to_categorical(y_test, num_classes)

x_train = x_train.reshape(60000, 784)
x_test = x_test.reshape(10000, 784)


In [2]:
REPO_NAME= "Curso-de-redes-neuronales-FCFM"
REPO_OWNER= "Oscar-Eduardo-Gonzalez-Jaramillo"  #Escribir nombre de repositorio
USER_NAME = "Oscar-Eduardo-Gonzalez-Jaramillo" #Escribir su usuario

In [3]:
os.environ['MLFLOW_TRACKING_USERNAME'] = USER_NAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = getpass('Enter your DAGsHub access token or password: ')
mlflow.set_tracking_uri(f'https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow')


In [4]:
model = Sequential()

model.add(Dense(100, activation='relu', input_shape=(784,)))
model.add(Dense(30, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))

c:\Users\Oscar\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model_dropout = Sequential()

model_dropout.add(Dense(100, activation='relu', input_shape=(784,)))
model_dropout.add(Dropout(0.2))
model_dropout.add(Dense(30, activation='relu'))
model_dropout.add(Dropout(0.2))
model_dropout.add(Dense(num_classes, activation='softmax'))


In [ ]:
model1l2_dropout = Sequential()

model1l2_dropout.add(Dense(100, activation='relu', input_shape=(784,), kernel_regularizer=l1_l2(0.01,0.01)))
model_dropout.add(Dropout(0.2))
model1l2_dropout.add(Dense(30, activation='relu', kernel_regularizer=l1_l2(0.01,0.01)))
model_dropout.add(Dropout(0.2))
model1l2_dropout.add(Dense(num_classes, activation='softmax'))

In [ ]:

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 30)             │         3,030 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           310 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 81,840 (319.69 KB)

 Trainable params: 81,840 (319.69 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
import numpy as np 
learning_rates = [0.0001, 0.0005, 0.001]
batch_sizes = [16, 32, 64, 128, 256, 512]


In [ ]:
from tensorflow.keras.models import clone_model
import mlflow

mlflow.tensorflow.autolog(log_models=True)
mlflow.set_experiment("Network_mejorada_784_100_30_10")

for lr in learning_rates:
    for bs in batch_sizes:
        with mlflow.start_run() as run:
            model = clone_model(model)  
            
            earlystop = EarlyStopping(
                monitor='val_loss',
                mode='min',
                restore_best_weights=True,
                patience=10,
                verbose=1
            )
            model.compile(
                loss="categorical_crossentropy",
                optimizer=Adam(learning_rate=lr),
                metrics=['accuracy']
            )
            history = model.fit(
                x_train,
                y_trainc,
                batch_size=bs,
                epochs=300,
                verbose=1,
                validation_data=(x_test, y_testc),
                callbacks=[earlystop]
            )

            # Guarda el modelo con un nombre único para cada combinación (opcional, para evitar sobrescribir)
            model_path = f"mi_modelo_keras_lr_{lr}_bs_{bs}.keras"
            model.save(model_path)
            print(f"Modelo guardado en: {model_path}")
            mlflow.log_artifact(model_path, artifact_path="model")

2025/09/15 18:43:31 WARNING mlflow.utils.autologging_utils: MLflow tensorflow autologging is known to be compatible with 2.7.4 <= tensorflow <= 2.19.0, but the installed version is 2.20.0. If you encounter errors during autologging, try upgrading / downgrading tensorflow to a compatible version, or try upgrading MLflow.


Epoch 1/300
3745/3750 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7359 - loss: 0.9420

3750/3750 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.8571 - loss: 0.5376 - val_accuracy: 0.9272 - val_loss: 0.2621
Epoch 2/300
3746/3750 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9305 - loss: 0.2503

3750/3750 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9332 - loss: 0.2354 - val_accuracy: 0.9437 - val_loss: 0.1976
Epoch 3/300
3722/3750 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9426 - loss: 0.1985

3750/3750 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9474 - loss: 0.1856 - val_accuracy: 0.9519 - val_loss: 0.1680
Epoch 4/300
3724/3750 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9538 - loss: 0.1603

...

118/118 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step - accuracy: 0.9824 - loss: 0.0607 - val_accuracy: 0.9710 - val_loss: 0.0940
Epoch 12/300
104/118 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9834 - loss: 0.0560

118/118 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step - accuracy: 0.9838 - loss: 0.0554 - val_accuracy: 0.9724 - val_loss: 0.0887
Epoch 13/300
111/118 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9868 - loss: 0.0487

118/118 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step - accuracy: 0.9864 - loss: 0.0483 - val_accuracy: 0.9748 - val_loss: 0.0836
Epoch 14/300
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9872 - loss: 0.0443 - val_accuracy: 0.9751 - val_loss: 0.0846
Epoch 15/300
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9890 - loss: 0.0406 - val_accuracy: 0.9736 - val_loss: 0.0867
Epoch 16/300
109/118 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9895 - loss: 0.0384

118/118 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - accuracy: 0.9893 - loss: 0.0373 - val_accuracy: 0.9757 - val_loss: 0.0807
Epoch 17/300
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9912 - loss: 0.0327 - val_accuracy: 0.9735 - val_loss: 0.0873
Epoch 18/300
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9919 - loss: 0.0300 - val_accuracy: 0.9751 - val_loss: 0.0820
Epoch 19/300
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9930 - loss: 0.0271 - val_accuracy: 0.9735 - val_loss: 0.0866
Epoch 20/300
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9933 - loss: 0.0258 - val_accuracy: 0.9740 - val_loss: 0.0833
Epoch 21/300
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9947 - loss: 0.0223 - val_accuracy: 0.9749 - val_loss: 0.0854
Epoch 22/300
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9952 - loss: 0.0208 - val_accuracy: 0.9758 - val_loss: 0.0847
Epoch 23/300
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9958 - loss: 0.0187 - val_accuracy

Modelo guardado en: mi_modelo_keras_lr_0.001_bs_512.keras
🏃 View run nimble-carp-874 at: https://dagshub.com/Oscar-Eduardo-Gonzalez-Jaramillo/Curso-de-redes-neuronales-FCFM.mlflow/#/experiments/8/runs/8239593c04c54bf698ba701a73180ddf
🧪 View experiment at: https://dagshub.com/Oscar-Eduardo-Gonzalez-Jaramillo/Curso-de-redes-neuronales-FCFM.mlflow/#/experiments/8


In [30]:
learning_rates = [0.0001, 0.0005, 0.001]
batch_sizes_filtrados = [32, 64, 256]
lambda_l1 = [0.0001, 0.0005, 0.001]
lambda_l2 = [0.001, 0.01, 0.1]
dropout = [0.1, 0.2, 0.3]

In [28]:


mlflow.tensorflow.autolog(log_models=True)
mlflow.set_experiment("Network_regularizada_l1_784_100_30_10")  
for k in lambda_l1:
    
    model1 = Sequential()
    model1.add(Dense(100, activation='relu', input_shape=(784,), kernel_regularizer=l1(k))) 
    model1.add(Dense(30, activation='relu', kernel_regularizer=l1(k)))  
    model1.add(Dense(num_classes, activation='softmax'))
    
    for lr in learning_rates:
        for bs in batch_sizes_filtrados: 
            with mlflow.start_run() as run:
                
                mlflow.log_param("lambda_l1", k)
                
                
                model1_cloned = clone_model(model1)  
                
                earlystop = EarlyStopping(
                    monitor='val_loss',
                    mode='min',
                    restore_best_weights=True,
                    patience=10,
                    verbose=1
                )
                
                model1_cloned.compile(
                    loss="categorical_crossentropy",
                    optimizer=Adam(learning_rate=lr),
                    metrics=['accuracy']
                )
                
                history = model1_cloned.fit(
                    x_train,
                    y_trainc,
                    batch_size=bs,
                    epochs=300,
                    verbose=1,
                    validation_data=(x_test, y_testc),
                    callbacks=[earlystop]
                )

                
                model_path = f"mi_modelo_keras_l1_{k}_lr_{lr}_bs_{bs}.keras"
                model1_cloned.save(model_path)
                print(f"Modelo guardado en: {model_path}")
                mlflow.log_artifact(model_path, artifact_path="model")

2025/09/15 20:25:51 WARNING mlflow.utils.autologging_utils: MLflow tensorflow autologging is known to be compatible with 2.7.4 <= tensorflow <= 2.19.0, but the installed version is 2.20.0. If you encounter errors during autologging, try upgrading / downgrading tensorflow to a compatible version, or try upgrading MLflow.
2025/09/15 20:26:11 INFO mlflow.tracking.fluent: Experiment with name 'Network_regularizada_l1_784_100_30_10' does not exist. Creating a new experiment.
c:\Users\Oscar\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/300
1840/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6758 - loss: 1.4375

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - accuracy: 0.8195 - loss: 0.9388 - val_accuracy: 0.9133 - val_loss: 0.5440
Epoch 2/300
1850/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9129 - loss: 0.5256

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9183 - loss: 0.5024 - val_accuracy: 0.9281 - val_loss: 0.4529
Epoch 3/300
1847/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9320 - loss: 0.4418

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9328 - loss: 0.4330 - val_accuracy: 0.9379 - val_loss: 0.4062
Epoch 4/300
1869/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9419 - loss: 0.3949

...

235/235 ━━━━━━━━━━━━━━━━━━━━ 18s 76ms/step - accuracy: 0.9630 - loss: 0.2692 - val_accuracy: 0.9619 - val_loss: 0.2712
Epoch 118/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9629 - loss: 0.2700 - val_accuracy: 0.9563 - val_loss: 0.2815
Epoch 119/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9642 - loss: 0.2662 - val_accuracy: 0.9623 - val_loss: 0.2719
Epoch 120/300
221/235 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9645 - loss: 0.2659

235/235 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - accuracy: 0.9637 - loss: 0.2672 - val_accuracy: 0.9616 - val_loss: 0.2707
Epoch 121/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9642 - loss: 0.2670 - val_accuracy: 0.9608 - val_loss: 0.2726
Epoch 122/300
223/235 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9647 - loss: 0.2647

235/235 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step - accuracy: 0.9632 - loss: 0.2683 - val_accuracy: 0.9627 - val_loss: 0.2682
Epoch 123/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9640 - loss: 0.2659 - val_accuracy: 0.9621 - val_loss: 0.2730
Epoch 124/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9635 - loss: 0.2663 - val_accuracy: 0.9594 - val_loss: 0.2739
Epoch 125/300
222/235 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9653 - loss: 0.2637

235/235 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - accuracy: 0.9635 - loss: 0.2668 - val_accuracy: 0.9634 - val_loss: 0.2675
Epoch 126/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9649 - loss: 0.2616 - val_accuracy: 0.9624 - val_loss: 0.2698
Epoch 127/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9644 - loss: 0.2634 - val_accuracy: 0.9599 - val_loss: 0.2695
Epoch 128/300
223/235 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9638 - loss: 0.2650

235/235 ━━━━━━━━━━━━━━━━━━━━ 8s 32ms/step - accuracy: 0.9637 - loss: 0.2640 - val_accuracy: 0.9640 - val_loss: 0.2631
Epoch 129/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9644 - loss: 0.2626 - val_accuracy: 0.9609 - val_loss: 0.2691
Epoch 130/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9644 - loss: 0.2640 - val_accuracy: 0.9611 - val_loss: 0.2686
Epoch 131/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9653 - loss: 0.2604 - val_accuracy: 0.9569 - val_loss: 0.2819
Epoch 132/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9633 - loss: 0.2669 - val_accuracy: 0.9607 - val_loss: 0.2701
Epoch 133/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9641 - loss: 0.2639 - val_accuracy: 0.9608 - val_loss: 0.2721
Epoch 134/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9639 - loss: 0.2624 - val_accuracy: 0.9600 - val_loss: 0.2716
Epoch 135/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9646 - loss: 0.2628 - val_a

In [32]:
mlflow.tensorflow.autolog(log_models=True)
mlflow.set_experiment("Network_regularizada_l2_784_100_30_10")  
for k in lambda_l2:
    
    model2 = Sequential()
    model2.add(Dense(100, activation='relu', input_shape=(784,), kernel_regularizer=l2(k))) 
    model2.add(Dense(30, activation='relu', kernel_regularizer=l2(k)))  
    model2.add(Dense(num_classes, activation='softmax'))
    
    for lr in learning_rates:
        for bs in batch_sizes_filtrados: 
            with mlflow.start_run() as run:
                
                mlflow.log_param("lambda_l2", k)
                
                
                model2_cloned = clone_model(model2)  
                
                earlystop = EarlyStopping(
                    monitor='val_loss',
                    mode='min',
                    restore_best_weights=True,
                    patience=10,
                    verbose=1
                )
                
                model2_cloned.compile(
                    loss="categorical_crossentropy",
                    optimizer=Adam(learning_rate=lr),
                    metrics=['accuracy']
                )
                
                history = model2_cloned.fit(
                    x_train,
                    y_trainc,
                    batch_size=bs,
                    epochs=300,
                    verbose=1,
                    validation_data=(x_test, y_testc),
                    callbacks=[earlystop]
                )

                
                model_path = f"mi_modelo_keras_l2_{k}_lr_{lr}_bs_{bs}.keras"
                model1_cloned.save(model_path)
                print(f"Modelo guardado en: {model_path}")
                mlflow.log_artifact(model_path, artifact_path="model")
                

2025/09/16 00:22:27 WARNING mlflow.utils.autologging_utils: MLflow tensorflow autologging is known to be compatible with 2.7.4 <= tensorflow <= 2.19.0, but the installed version is 2.20.0. If you encounter errors during autologging, try upgrading / downgrading tensorflow to a compatible version, or try upgrading MLflow.


Epoch 1/300
1850/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6595 - loss: 1.4063

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8147 - loss: 0.8840 - val_accuracy: 0.9164 - val_loss: 0.4667
Epoch 2/300
1873/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9198 - loss: 0.4490

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9232 - loss: 0.4283 - val_accuracy: 0.9356 - val_loss: 0.3787
Epoch 3/300
1842/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9345 - loss: 0.3775

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9366 - loss: 0.3664 - val_accuracy: 0.9432 - val_loss: 0.3390
Epoch 4/300
1865/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9448 - loss: 0.3344

...

235/235 ━━━━━━━━━━━━━━━━━━━━ 9s 38ms/step - accuracy: 0.9340 - loss: 0.4268 - val_accuracy: 0.9352 - val_loss: 0.4171
Epoch 188/300
232/235 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9347 - loss: 0.4254

235/235 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.9336 - loss: 0.4274 - val_accuracy: 0.9392 - val_loss: 0.4080
Epoch 189/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9350 - loss: 0.4242 - val_accuracy: 0.9397 - val_loss: 0.4088
Epoch 190/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9344 - loss: 0.4232 - val_accuracy: 0.9400 - val_loss: 0.4095
Epoch 191/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9328 - loss: 0.4253 - val_accuracy: 0.9346 - val_loss: 0.4255
Epoch 192/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9343 - loss: 0.4226 - val_accuracy: 0.9348 - val_loss: 0.4130
Epoch 193/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9347 - loss: 0.4229 - val_accuracy: 0.9348 - val_loss: 0.4084
Epoch 194/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9349 - loss: 0.4209 - val_accuracy: 0.9362 - val_loss: 0.4133
Epoch 195/300
226/235 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9345 - loss: 0.4226

235/235 ━━━━━━━━━━━━━━━━━━━━ 11s 47ms/step - accuracy: 0.9350 - loss: 0.4214 - val_accuracy: 0.9400 - val_loss: 0.4069
Epoch 196/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9341 - loss: 0.4214 - val_accuracy: 0.9359 - val_loss: 0.4131
Epoch 197/300
223/235 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9322 - loss: 0.4274

235/235 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - accuracy: 0.9334 - loss: 0.4237 - val_accuracy: 0.9397 - val_loss: 0.4043
Epoch 198/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9347 - loss: 0.4221 - val_accuracy: 0.9360 - val_loss: 0.4075
Epoch 199/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9354 - loss: 0.4183 - val_accuracy: 0.9367 - val_loss: 0.4047
Epoch 200/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9348 - loss: 0.4199 - val_accuracy: 0.9284 - val_loss: 0.4282
Epoch 201/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9348 - loss: 0.4192 - val_accuracy: 0.9325 - val_loss: 0.4117
Epoch 202/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9352 - loss: 0.4176 - val_accuracy: 0.9380 - val_loss: 0.4192
Epoch 203/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9348 - loss: 0.4164 - val_accuracy: 0.9361 - val_loss: 0.4082
Epoch 204/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9338 - loss: 0.4193 - val_a

Modelo guardado en: mi_modelo_keras_l2_0.1_lr_0.001_bs_256.keras
🏃 View run mysterious-zebra-454 at: https://dagshub.com/Oscar-Eduardo-Gonzalez-Jaramillo/Curso-de-redes-neuronales-FCFM.mlflow/#/experiments/11/runs/885bd1a7aef447fe9bc03322f61c6ea5
🧪 View experiment at: https://dagshub.com/Oscar-Eduardo-Gonzalez-Jaramillo/Curso-de-redes-neuronales-FCFM.mlflow/#/experiments/11


In [ ]:
learning_rates = [0.0001, 0.0005, 0.001]
batch_sizes_filtrados = [32, 64, 256]
lambda_filtrado = [[0.001, 0.01], [0.0001, 0.1]]


In [37]:
mlflow.tensorflow.autolog(log_models=True)
mlflow.set_experiment("Network_regularizada_l1_l2_784_100_30_10")  
for k, l in lambda_filtrado:
    model1l2 = Sequential()
    model1l2.add(Dense(100, activation='relu', input_shape=(784,), kernel_regularizer=l1_l2(k,l))) 
    model1l2.add(Dense(30, activation='relu', kernel_regularizer=l1_l2(k,l)))  
    model1l2.add(Dense(num_classes, activation='softmax'))
    for lr in learning_rates:
        for bs in batch_sizes_filtrados: 
            with mlflow.start_run() as run:
                mlflow.log_param("lambda_l1", k)
                mlflow.log_param("lambda_l2", l)
                
                
                model2_cloned = clone_model(model1l2)  
                
                earlystop = EarlyStopping(
                    monitor='val_loss',
                    mode='min',
                    restore_best_weights=True,
                    patience=10,
                    verbose=1
                )
                
                model2_cloned.compile(
                    loss="categorical_crossentropy",
                    optimizer=Adam(learning_rate=lr),
                    metrics=['accuracy']
                )
                
                history = model2_cloned.fit(
                    x_train,
                    y_trainc,
                    batch_size=bs,
                    epochs=300,
                    verbose=1,
                    validation_data=(x_test, y_testc),
                    callbacks=[earlystop]
                )

                
                model_path = f"mi_modelo_keras_l1_{k}_l2_{l}_lr_{lr}_bs_{bs}.keras"
                model1_cloned.save(model_path)
                print(f"Modelo guardado en: {model_path}")
                mlflow.log_artifact(model_path, artifact_path="model")




2025/09/16 13:03:14 WARNING mlflow.utils.autologging_utils: MLflow tensorflow autologging is known to be compatible with 2.7.4 <= tensorflow <= 2.19.0, but the installed version is 2.20.0. If you encounter errors during autologging, try upgrading / downgrading tensorflow to a compatible version, or try upgrading MLflow.
c:\Users\Oscar\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/300
1868/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6565 - loss: 4.6038

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 25s 13ms/step - accuracy: 0.7994 - loss: 2.9529 - val_accuracy: 0.8851 - val_loss: 1.4824
Epoch 2/300
1871/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8839 - loss: 1.3771

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8865 - loss: 1.2852 - val_accuracy: 0.8898 - val_loss: 1.1315
Epoch 3/300
1846/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8914 - loss: 1.1056

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8907 - loss: 1.0770 - val_accuracy: 0.8946 - val_loss: 1.0008
Epoch 4/300
1873/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8944 - loss: 0.9935

...

235/235 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.9200 - loss: 0.4924 - val_accuracy: 0.9227 - val_loss: 0.4720
Epoch 131/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9186 - loss: 0.4954 - val_accuracy: 0.9184 - val_loss: 0.4876
Epoch 132/300
229/235 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9224 - loss: 0.4853

235/235 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - accuracy: 0.9209 - loss: 0.4905 - val_accuracy: 0.9249 - val_loss: 0.4719
Epoch 133/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9202 - loss: 0.4911 - val_accuracy: 0.9236 - val_loss: 0.4803
Epoch 134/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9204 - loss: 0.4909 - val_accuracy: 0.9265 - val_loss: 0.4776
Epoch 135/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9198 - loss: 0.4911 - val_accuracy: 0.9234 - val_loss: 0.4736
Epoch 136/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9197 - loss: 0.4874 - val_accuracy: 0.9230 - val_loss: 0.4766
Epoch 137/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9216 - loss: 0.4860 - val_accuracy: 0.9229 - val_loss: 0.4722
Epoch 138/300
233/235 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9188 - loss: 0.4880

235/235 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.9207 - loss: 0.4859 - val_accuracy: 0.9279 - val_loss: 0.4698
Epoch 139/300
227/235 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9230 - loss: 0.4798

235/235 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.9218 - loss: 0.4850 - val_accuracy: 0.9261 - val_loss: 0.4623
Epoch 140/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9212 - loss: 0.4830 - val_accuracy: 0.9241 - val_loss: 0.4714
Epoch 141/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9204 - loss: 0.4854 - val_accuracy: 0.9233 - val_loss: 0.4811
Epoch 142/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9212 - loss: 0.4832 - val_accuracy: 0.9269 - val_loss: 0.4682
Epoch 143/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9207 - loss: 0.4844 - val_accuracy: 0.9179 - val_loss: 0.4781
Epoch 144/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9208 - loss: 0.4840 - val_accuracy: 0.9247 - val_loss: 0.4644
Epoch 145/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9217 - loss: 0.4809 - val_accuracy: 0.9275 - val_loss: 0.4636
Epoch 146/300
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9208 - loss: 0.4809 - val_a

Modelo guardado en: mi_modelo_keras_l1_0.0001_l2_0.1_lr_0.001_bs_256.keras
🏃 View run judicious-carp-998 at: https://dagshub.com/Oscar-Eduardo-Gonzalez-Jaramillo/Curso-de-redes-neuronales-FCFM.mlflow/#/experiments/12/runs/d996734e0353476eb4dad72c056acd00
🧪 View experiment at: https://dagshub.com/Oscar-Eduardo-Gonzalez-Jaramillo/Curso-de-redes-neuronales-FCFM.mlflow/#/experiments/12


In [38]:
learning_rates = [0.0001, 0.0005, 0.001]
batch_sizes_filtrados = [32, 64, 256]
dropout = [0.1, 0.2, 0.3]

In [40]:
mlflow.tensorflow.autolog(log_models=True)
mlflow.set_experiment("Network_regularizada_dropout_784_100_30_10")  
for d in dropout:
    model_dropout = Sequential()
    model_dropout.add(Dense(100, activation='relu', input_shape=(784,))) 
    model_dropout.add(Dropout(d))
    model_dropout.add(Dense(30, activation='relu'))
    model_dropout.add(Dropout(d))  
    model_dropout.add(Dense(num_classes, activation='softmax'))
    for lr in learning_rates:
        for bs in batch_sizes_filtrados: 
            with mlflow.start_run() as run:
                mlflow.log_param("dropout", d)
                
                model2_cloned = clone_model(model1l2)  
                
                earlystop = EarlyStopping(
                    monitor='val_loss',
                    mode='min',
                    restore_best_weights=True,
                    patience=10,
                    verbose=1
                )
                
                model2_cloned.compile(
                    loss="categorical_crossentropy",
                    optimizer=Adam(learning_rate=lr),
                    metrics=['accuracy']
                )
                
                history = model2_cloned.fit(
                    x_train,
                    y_trainc,
                    batch_size=bs,
                    epochs=150,
                    verbose=1,
                    validation_data=(x_test, y_testc),
                    callbacks=[earlystop]
                )

                
                model_path = f"mi_modelo_keras_l1_dropout_{d}_lr_{lr}_bs_{bs}.keras"
                model1_cloned.save(model_path)
                print(f"Modelo guardado en: {model_path}")
                mlflow.log_artifact(model_path, artifact_path="model")

2025/09/16 17:14:31 WARNING mlflow.utils.autologging_utils: MLflow tensorflow autologging is known to be compatible with 2.7.4 <= tensorflow <= 2.19.0, but the installed version is 2.20.0. If you encounter errors during autologging, try upgrading / downgrading tensorflow to a compatible version, or try upgrading MLflow.


Epoch 1/150
1858/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5653 - loss: 11.3756

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.7229 - loss: 5.9812 - val_accuracy: 0.8055 - val_loss: 1.9218
Epoch 2/150
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8117 - loss: 1.7962

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8117 - loss: 1.6942 - val_accuracy: 0.8110 - val_loss: 1.5338
Epoch 3/150
1855/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8121 - loss: 1.5203

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8166 - loss: 1.4851 - val_accuracy: 0.8285 - val_loss: 1.4015
Epoch 4/150
1853/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8227 - loss: 1.4022

...

235/235 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - accuracy: 0.9250 - loss: 0.4885 - val_accuracy: 0.9291 - val_loss: 0.4719
Epoch 120/150
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9245 - loss: 0.4871 - val_accuracy: 0.9268 - val_loss: 0.4778
Epoch 121/150
228/235 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9222 - loss: 0.4932

235/235 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.9239 - loss: 0.4886 - val_accuracy: 0.9276 - val_loss: 0.4717
Epoch 122/150
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9249 - loss: 0.4856 - val_accuracy: 0.9283 - val_loss: 0.4774
Epoch 123/150
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9260 - loss: 0.4838 - val_accuracy: 0.9292 - val_loss: 0.4725
Epoch 124/150
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9253 - loss: 0.4832 - val_accuracy: 0.9216 - val_loss: 0.5010
Epoch 125/150
225/235 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9254 - loss: 0.4828

235/235 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - accuracy: 0.9246 - loss: 0.4846 - val_accuracy: 0.9318 - val_loss: 0.4673
Epoch 126/150
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9245 - loss: 0.4835 - val_accuracy: 0.9295 - val_loss: 0.4708
Epoch 127/150
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9255 - loss: 0.4813 - val_accuracy: 0.9270 - val_loss: 0.4680
Epoch 128/150
228/235 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9266 - loss: 0.4790

235/235 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - accuracy: 0.9259 - loss: 0.4800 - val_accuracy: 0.9293 - val_loss: 0.4611
Epoch 129/150
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9252 - loss: 0.4798 - val_accuracy: 0.9308 - val_loss: 0.4652
Epoch 130/150
230/235 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9251 - loss: 0.4768

235/235 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step - accuracy: 0.9238 - loss: 0.4805 - val_accuracy: 0.9305 - val_loss: 0.4574
Epoch 131/150
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9252 - loss: 0.4779 - val_accuracy: 0.9278 - val_loss: 0.4671
Epoch 132/150
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9254 - loss: 0.4769 - val_accuracy: 0.9330 - val_loss: 0.4609
Epoch 133/150
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9259 - loss: 0.4758 - val_accuracy: 0.9309 - val_loss: 0.4622
Epoch 134/150
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9260 - loss: 0.4771 - val_accuracy: 0.9293 - val_loss: 0.4590
Epoch 135/150
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9259 - loss: 0.4744 - val_accuracy: 0.9317 - val_loss: 0.4587
Epoch 136/150
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9262 - loss: 0.4735 - val_accuracy: 0.9318 - val_loss: 0.4576
Epoch 137/150
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9257 - loss: 0.4752 - val_a

In [41]:
learning_rates = [0.0001]
batch_sizes_filtrados = [32]
lambda_filtrado = [[0.001, 0.01]]
dropout = [0.1]

In [42]:
mlflow.tensorflow.autolog(log_models=True)
mlflow.set_experiment("Network_regularizada_l1_l2_dropout_784_100_30_10")  
for k, l, d in (lambda_l1, lambda_l2, dropout):
    model1l2 = Sequential()
    model1l2.add(Dense(100, activation='relu', input_shape=(784,), kernel_regularizer=l1_l2(k,l)))
    model_dropout.add(Dropout(d))
    model1l2.add(Dense(30, activation='relu', kernel_regularizer=l1_l2(k,l)))  
    model_dropout.add(Dropout(d))
    model1l2.add(Dense(num_classes, activation='softmax'))
    for lr in learning_rates:
        for bs in batch_sizes_filtrados: 
            with mlflow.start_run() as run:
                mlflow.log_param("lambda_l1", k)
                mlflow.log_param("lambda_l2", l)
                mlflow.log_param("Porcentaje_dropout", d)
                
                model2_cloned = clone_model(model1l2)  
                
                earlystop = EarlyStopping(
                    monitor='val_loss',
                    mode='min',
                    restore_best_weights=True,
                    patience=10,
                    verbose=1
                )
                
                model2_cloned.compile(
                    loss="categorical_crossentropy",
                    optimizer=Adam(learning_rate=lr),
                    metrics=['accuracy']
                )
                
                history = model2_cloned.fit(
                    x_train,
                    y_trainc,
                    batch_size=bs,
                    epochs=100,
                    verbose=1,
                    validation_data=(x_test, y_testc),
                    callbacks=[earlystop]
                )

                
                model_path = f"mi_modelo_keras_l1_{k}_l2_{l}_dropout_{d}_lr_{lr}_bs_{bs}.keras"
                model1_cloned.save(model_path)
                print(f"Modelo guardado en: {model_path}")
                mlflow.log_artifact(model_path, artifact_path="model")


2025/09/16 22:11:35 WARNING mlflow.utils.autologging_utils: MLflow tensorflow autologging is known to be compatible with 2.7.4 <= tensorflow <= 2.19.0, but the installed version is 2.20.0. If you encounter errors during autologging, try upgrading / downgrading tensorflow to a compatible version, or try upgrading MLflow.
2025/09/16 22:11:36 INFO mlflow.tracking.fluent: Experiment with name 'Network_regularizada_l1_l2_dropout_784_100_30_10' does not exist. Creating a new experiment.


Epoch 1/100
1860/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7097 - loss: 1.4544

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.8380 - loss: 0.9642 - val_accuracy: 0.9172 - val_loss: 0.5961
Epoch 2/100
1865/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9158 - loss: 0.5775

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9206 - loss: 0.5526 - val_accuracy: 0.9325 - val_loss: 0.4954
Epoch 3/100
1871/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9322 - loss: 0.4894

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.9333 - loss: 0.4791 - val_accuracy: 0.9405 - val_loss: 0.4469
Epoch 4/100
1870/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9398 - loss: 0.4416

...

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9405 - loss: 0.4025 - val_accuracy: 0.9393 - val_loss: 0.3987
Epoch 98/100
1858/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9410 - loss: 0.4001

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9418 - loss: 0.4018 - val_accuracy: 0.9397 - val_loss: 0.3969
Epoch 99/100
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9413 - loss: 0.4037

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9416 - loss: 0.4012 - val_accuracy: 0.9414 - val_loss: 0.3939
Epoch 100/100
1865/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9405 - loss: 0.4003

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9405 - loss: 0.4001 - val_accuracy: 0.9417 - val_loss: 0.3932
Restoring model weights from the end of the best epoch: 100.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
Modelo guardado en: mi_modelo_keras_l1_0.001_l2_0.01_dropout_0.1_lr_0.0001_bs_32.keras
🏃 View run masked-dolphin-408 at: https://dagshub.com/Oscar-Eduardo-Gonzalez-Jaramillo/Curso-de-redes-neuronales-FCFM.mlflow/#/experiments/14/runs/24be76cd5857403ba755a006af69393e
🧪 View experiment at: https://dagshub.com/Oscar-Eduardo-Gonzalez-Jaramillo/Curso-de-redes-neuronales-FCFM.mlflow/#/experiments/14


ValueError: not enough values to unpack (expected 3, got 1)

In [ ]:
print("Mejor época según EarlyStopping:", earlystop.best_epoch)
print("Mejor val_loss registrado:", earlystop.best)
a = np.round(9.999999747378752e-05, 7)
print(a)

Mejor época según EarlyStopping: 15
Mejor val_loss registrado: 0.0806848481297493
0.0001
